# Compliance Copilot - October milestone (Ollama)
For local development in VS Code, use `scripts/compliance_copilot_october_ollama.py`.
This notebook remains available for Colab experiments. Use a T4 GPU if available.
The modular implementation requires the repository's `src` package, not only the baseline script.


## 1. Install Ollama (one-time per runtime)

In [ ]:
%%bash
set -euo pipefail
if ! command -v zstd >/dev/null 2>&1; then
  apt-get update --quiet
  DEBIAN_FRONTEND=noninteractive apt-get install -y zstd
fi
if ! command -v ollama >/dev/null 2>&1 || \
   ! ollama --version 2>&1 | grep -Eq '(^|[^0-9.])0\.33\.3([^0-9.]|$)'; then
  curl -fsSL https://ollama.com/install.sh | OLLAMA_VERSION=0.33.3 sh
fi
python -m pip install --quiet ollama==0.6.2 pydantic==2.13.5 numpy pandas requests tabulate faiss-cpu
ollama --version

## 2. Start the server and pull the models

In [ ]:
import os, subprocess, time, requests
from pathlib import Path

GENERATION_MODEL = "qwen3:4b"   # optional upgrade on a T4: "qwen3:8b" (re-run this cell to pull it)
EMBEDDING_MODEL  = "nomic-embed-text"
HOST = "http://127.0.0.1:11434"
os.environ["OLLAMA_HOST"] = HOST          # the script reads this

def ready():
    try: return requests.get(f"{HOST}/api/tags", timeout=2).ok
    except requests.RequestException: return False

if not ready():
    log = open("/tmp/ollama-colab.log", "w")
    subprocess.Popen(["ollama", "serve"], stdout=log, stderr=subprocess.STDOUT,
                     env={**os.environ, "OLLAMA_HOST": "127.0.0.1:11434"})
t_end = time.monotonic() + 60
while not ready() and time.monotonic() < t_end:
    time.sleep(1)
assert ready(), "Ollama did not start:\n" + Path("/tmp/ollama-colab.log").read_text()[-3000:]
print("Ollama server ready.")

for m in (EMBEDDING_MODEL, GENERATION_MODEL):
    print("pulling", m); subprocess.run(["ollama", "pull", m], check=True)

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "No GPU detected - this will be slow"

## 3. Load repository code and data
Clone the modularization branch (or set `REPO` to an existing checkout).
The repository includes the clause corpus and benchmark. Add additional benchmark CSVs to `DATA` if desired.


In [ ]:
import subprocess, sys
from pathlib import Path

REPO = Path("/content/Aerospace-1D-compliance-copilot")
if not REPO.exists():
    subprocess.run(["git", "clone", "--branch", "modulzarizing-code",
                    "https://github.com/Break-Through-Tech/Aerospace-1D-compliance-copilot.git",
                    str(REPO)], check=True)
sys.path.insert(0, str(REPO))
DATA = str(REPO / "data")
print("data files:", sorted(p.name for p in Path(DATA).glob("*.csv")))


## 4. Auto-detect files by their columns
- **Clause corpus**: has `clause` + `requirement` columns
- **Benchmark**: has `requirement_text`, `clause_id`, `label`. Multiple benchmark files are combined (duplicates dropped).

In [ ]:
from src.compliance_rag.data import discover_csvs, combine_benchmarks
from src.compliance_rag import cli as crb

clauses_csv, bench_files = discover_csvs(DATA)
assert clauses_csv and bench_files, "No compatible corpus/benchmark CSVs found"
bench = combine_benchmarks(bench_files)
COMBINED = "/content/data_combined_benchmark.csv"
bench.to_csv(COMBINED, index=False)
print("clauses:", clauses_csv)
print("benchmark rows:", len(bench))


## 5. Run the baseline
Dry run first with `LIMIT = 5`, then `LIMIT = None` for all rows. Finished rows are cached, so re-running resumes.

In [ ]:
RUN_NAME = "v2"    # change per experiment so results never overwrite each other (e.g. "v1_qwen4b", "v2_qwen8b")
LIMIT = 5          # set to None for the full run
K = 5              # clauses retrieved per requirement
# Cache survives a runtime restart only if it lives on Drive, e.g.
OUT = f"/content/out_{RUN_NAME}"
# CACHE = f"/content/drive/MyDrive/compliance_copilot/cache_{RUN_NAME}.jsonl"
CACHE = f"{OUT}/cache.jsonl"

args = ["--clauses", str(clauses_csv), "--benchmark", COMBINED,
        "--embedder", "ollama", "--ollama-embed-model", EMBEDDING_MODEL,
        "--judge", "ollama", "--model", GENERATION_MODEL,
        "--k", str(K), "--out", OUT, "--cache", CACHE]
if LIMIT: args += ["--limit", str(LIMIT)]
df, metrics = crb.main(args)

## 6. Results

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open(f"{OUT}/report.md").read()))

In [ ]:
df[["requirement_id","requirement_text","label","pred_verdict","clause_id","pred_clause","pred_rationale"]].head(20)

## 7. Download outputs

In [ ]:
import shutil
from google.colab import files
shutil.make_archive(f"/content/october_results_{RUN_NAME}", "zip", OUT)
files.download(f"/content/october_results_{RUN_NAME}.zip")